# NicePool TypeScript scientific API

This notebook imports a generated browser-independent ESM build of the NicePool TypeScript engine. The bundle is built from `src/core` and is not a second implementation. It demonstrates dataset initialization, schema inspection, deterministic filtering and statistics, plot preparation, selection, and serializable state.

In [ ]:
import {
  DatasetStore,
  NicePoolEngine,
  descriptiveStatistics,
  filteredRowIndices,
} from "./nicepool-core.js";

## Define a typed NicePool dataset

In [ ]:
const input = {
  rowIdColumn: "pool_row_id",
  rows: [
    { pool_row_id: "a", accept: "yes", condition: "control", x: -2, y: 1 },
    { pool_row_id: "b", accept: "yes", condition: "control", x: -1, y: 3 },
    { pool_row_id: "c", accept: "no", condition: "treated", x: 1, y: 5 },
    { pool_row_id: "d", accept: "yes", condition: "treated", x: 2, y: null },
    { pool_row_id: "e", accept: "yes", condition: "treated", x: 3, y: 9 },
  ],
};
input

## Inspect the immutable dataset and inferred schema

In [ ]:
const dataset = new DatasetStore(input);
({
  rows: dataset.rows.length,
  schema: dataset.schema,
  numericColumns: dataset.numericColumns(),
  categoricalColumns: dataset.categoricalColumns(),
  conditions: dataset.uniqueValues("condition"),
})

## Apply a deterministic categorical prefilter

In [ ]:
const acceptedIndices = filteredRowIndices(dataset, { accept: "yes" });
const acceptedRows = acceptedIndices.map((index) => dataset.rows[index]);
acceptedRows

## Run the project-owned statistics implementation

In [ ]:
const acceptedY = acceptedRows
  .map((row) => row.y)
  .filter((value): value is number => typeof value === "number");
descriptiveStatistics(acceptedY)

## Drive the same engine used by the GUI

In [ ]:
const engine = new NicePoolEngine();
engine.setData(input);
engine.setPlotState({
  ...engine.plotState,
  plotType: "swarm",
  xColumn: "condition",
  yColumn: "y",
  groupColumn: "condition",
  preFilters: { accept: "yes" },
});
const prepared = engine.preparePlot();
prepared

## Selection and serializable analysis state

In [ ]:
engine.setSelection({ primaryRowId: "e", selectedRowIds: ["b", "e"] });
({
  selection: engine.selection,
  state: engine.state,
  stateJson: JSON.stringify(engine.state, null, 2),
})